# OctoCase - AI-Powered Test Case Generator

This notebook demonstrates how to automatically generate test cases for login functionality using AI/LLM technology.

## Features:
- Generate test cases using OpenAI or Hugging Face models
- Export results to Excel/CSV format
- Structured output with Test Case ID, Requirement, Step, Expected Result, and Priority


In [ ]:
# Import required libraries
import pandas as pd
import json
import os
import re
from datetime import datetime
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# For environment variables
from dotenv import load_dotenv
load_dotenv()

print("Libraries imported successfully!")

## Configuration

Set up your API keys and choose your preferred AI provider (OpenAI or Hugging Face).

In [ ]:
# Configuration
USE_OPENAI = True  # Set to False to use Hugging Face instead

# API Keys (can be set in .env file or directly here)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your_openai_api_key_here')
HUGGING_FACE_API_KEY = os.getenv('HUGGING_FACE_API_KEY', 'your_hugging_face_api_key_here')

# Output configuration
OUTPUT_FORMAT = 'excel'  # 'excel' or 'csv'
OUTPUT_FILENAME = f'test_cases_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print(f"Configuration set: Using {'OpenAI' if USE_OPENAI else 'Hugging Face'}")
print(f"Output format: {OUTPUT_FORMAT}")
print(f"Output filename: {OUTPUT_FILENAME}")

## Test Case Generator Class

Core class that handles test case generation using AI models.

In [ ]:
class TestCaseGenerator:
    def __init__(self, use_openai=True, openai_key=None, hf_key=None):
        self.use_openai = use_openai
        self.openai_key = openai_key
        self.hf_key = hf_key
        
        if use_openai:
            self._setup_openai()
        else:
            self._setup_huggingface()
    
    def _setup_openai(self):
        """Setup OpenAI client"""
        try:
            import openai
            if self.openai_key and self.openai_key != 'your_openai_api_key_here':
                self.client = openai.OpenAI(api_key=self.openai_key)
                print("✅ OpenAI client initialized successfully")
            else:
                print("⚠️ OpenAI API key not provided. Please set OPENAI_API_KEY in .env file or pass it directly.")
                self.client = None
        except ImportError:
            print("❌ OpenAI library not installed. Run: pip install openai")
            self.client = None
    
    def _setup_huggingface(self):
        """Setup Hugging Face pipeline"""
        try:
            from transformers import pipeline
            # Using a smaller, efficient model for text generation
            self.pipeline = pipeline(
                "text-generation", 
                model="microsoft/DialoGPT-medium",
                max_length=512
            )
            print("✅ Hugging Face pipeline initialized successfully")
        except ImportError:
            print("❌ Transformers library not installed. Run: pip install transformers torch")
            self.pipeline = None
    
    def generate_test_cases_openai(self, requirement: str) -> List[Dict]:
        """Generate test cases using OpenAI"""
        if not self.client:
            return self._generate_fallback_test_cases(requirement)
        
        prompt = f"""
Generate comprehensive test cases for the following requirement:

Requirement: {requirement}

Please generate test cases in JSON format with the following structure:
[
  {{
    "test_case_id": "TC_001",
    "requirement": "Specific requirement being tested",
    "step": "Detailed test steps",
    "expected_result": "Expected outcome",
    "priority": "High/Medium/Low"
  }}
]

Include positive test cases, negative test cases, edge cases, and boundary conditions.
Generate at least 8-10 comprehensive test cases.
"""
        
        try:
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": "You are a QA expert specializing in creating comprehensive test cases. Always respond with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7
            )
            
            content = response.choices[0].message.content
            # Extract JSON from the response
            json_match = re.search(r'\[.*\]', content, re.DOTALL)
            if json_match:
                test_cases = json.loads(json_match.group())
                return test_cases
            else:
                print("⚠️ Could not parse JSON from OpenAI response. Using fallback.")
                return self._generate_fallback_test_cases(requirement)
                
        except Exception as e:
            print(f"❌ Error with OpenAI API: {e}")
            return self._generate_fallback_test_cases(requirement)
    
    def generate_test_cases_huggingface(self, requirement: str) -> List[Dict]:
        """Generate test cases using Hugging Face (with fallback logic)"""
        if not self.pipeline:
            return self._generate_fallback_test_cases(requirement)
        
        # For this demo, we'll use the fallback method as HF free models 
        # are not as effective for structured JSON generation
        print("ℹ️ Using rule-based generation (HF models work better with API access)")
        return self._generate_fallback_test_cases(requirement)
    
    def _generate_fallback_test_cases(self, requirement: str) -> List[Dict]:
        """Generate test cases using rule-based approach when AI is not available"""
        
        # Parse requirement to understand the functionality
        is_login = 'login' in requirement.lower() or 'log in' in requirement.lower()
        has_username = 'username' in requirement.lower()
        has_password = 'password' in requirement.lower()
        has_error = 'error' in requirement.lower()
        
        test_cases = []
        
        if is_login:
            # Generate login-specific test cases
            base_cases = [
                {
                    "test_case_id": "TC_001",
                    "requirement": "Valid user login with correct credentials",
                    "step": "1. Navigate to login page\n2. Enter valid username\n3. Enter valid password\n4. Click login button",
                    "expected_result": "User is successfully logged in and redirected to dashboard",
                    "priority": "High"
                },
                {
                    "test_case_id": "TC_002",
                    "requirement": "Invalid login with incorrect username",
                    "step": "1. Navigate to login page\n2. Enter invalid username\n3. Enter valid password\n4. Click login button",
                    "expected_result": "Error message displayed: 'Invalid username or password'",
                    "priority": "High"
                },
                {
                    "test_case_id": "TC_003",
                    "requirement": "Invalid login with incorrect password",
                    "step": "1. Navigate to login page\n2. Enter valid username\n3. Enter invalid password\n4. Click login button",
                    "expected_result": "Error message displayed: 'Invalid username or password'",
                    "priority": "High"
                },
                {
                    "test_case_id": "TC_004",
                    "requirement": "Login with empty username field",
                    "step": "1. Navigate to login page\n2. Leave username field empty\n3. Enter valid password\n4. Click login button",
                    "expected_result": "Error message displayed: 'Username is required'",
                    "priority": "Medium"
                },
                {
                    "test_case_id": "TC_005",
                    "requirement": "Login with empty password field",
                    "step": "1. Navigate to login page\n2. Enter valid username\n3. Leave password field empty\n4. Click login button",
                    "expected_result": "Error message displayed: 'Password is required'",
                    "priority": "Medium"
                },
                {
                    "test_case_id": "TC_006",
                    "requirement": "Login with both fields empty",
                    "step": "1. Navigate to login page\n2. Leave both username and password fields empty\n3. Click login button",
                    "expected_result": "Error messages displayed for both required fields",
                    "priority": "Medium"
                },
                {
                    "test_case_id": "TC_007",
                    "requirement": "Login with special characters in username",
                    "step": "1. Navigate to login page\n2. Enter username with special characters (!@#$%)\n3. Enter valid password\n4. Click login button",
                    "expected_result": "System handles special characters appropriately or shows validation error",
                    "priority": "Low"
                },
                {
                    "test_case_id": "TC_008",
                    "requirement": "Login with SQL injection attempt",
                    "step": "1. Navigate to login page\n2. Enter SQL injection string in username (e.g., ' OR '1'='1)\n3. Enter any password\n4. Click login button",
                    "expected_result": "Login fails securely, no SQL injection vulnerability",
                    "priority": "High"
                },
                {
                    "test_case_id": "TC_009",
                    "requirement": "Login with maximum length username",
                    "step": "1. Navigate to login page\n2. Enter username at maximum allowed length\n3. Enter valid password\n4. Click login button",
                    "expected_result": "System accepts or rejects based on validation rules",
                    "priority": "Low"
                },
                {
                    "test_case_id": "TC_010",
                    "requirement": "Case sensitivity test for username",
                    "step": "1. Navigate to login page\n2. Enter valid username in different case (upper/lower)\n3. Enter valid password\n4. Click login button",
                    "expected_result": "Login success/failure based on system case sensitivity rules",
                    "priority": "Medium"
                }
            ]
            test_cases.extend(base_cases)
        else:
            # Generic test cases for other requirements
            test_cases = [
                {
                    "test_case_id": "TC_001",
                    "requirement": f"Positive test case for: {requirement}",
                    "step": "1. Perform the required action with valid inputs\n2. Verify the functionality works as expected",
                    "expected_result": "The functionality works as specified in the requirement",
                    "priority": "High"
                },
                {
                    "test_case_id": "TC_002",
                    "requirement": f"Negative test case for: {requirement}",
                    "step": "1. Perform the required action with invalid inputs\n2. Verify error handling",
                    "expected_result": "Appropriate error message is displayed",
                    "priority": "High"
                }
            ]
        
        return test_cases
    
    def generate_test_cases(self, requirement: str) -> List[Dict]:
        """Main method to generate test cases"""
        print(f"Generating test cases for: {requirement}")
        
        if self.use_openai:
            return self.generate_test_cases_openai(requirement)
        else:
            return self.generate_test_cases_huggingface(requirement)

# Initialize the generator
generator = TestCaseGenerator(
    use_openai=USE_OPENAI,
    openai_key=OPENAI_API_KEY,
    hf_key=HUGGING_FACE_API_KEY
)

print("✅ TestCaseGenerator initialized successfully!")

## Generate Test Cases

Define your requirement and generate test cases.

In [ ]:
# Define the requirement (you can modify this)
login_requirement = "The user can log in with a valid username and password. If incorrect, an error message appears."

print(f"Requirement: {login_requirement}")
print("\nGenerating test cases...")

# Generate test cases
test_cases = generator.generate_test_cases(login_requirement)

print(f"\n✅ Generated {len(test_cases)} test cases successfully!")

# Display first few test cases
print("\n📋 Sample test cases:")
for i, tc in enumerate(test_cases[:3]):
    print(f"\n{i+1}. {tc['test_case_id']}: {tc['requirement']}")
    print(f"   Priority: {tc['priority']}")

## View All Generated Test Cases

Display all generated test cases in a readable format.

In [ ]:
# Create DataFrame for better visualization
df = pd.DataFrame(test_cases)

# Display the DataFrame
print("📊 All Generated Test Cases:")
print("=" * 80)

# Configure pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', None)

print(df.to_string(index=False))

# Reset display options
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')

## Export Test Cases

Export the generated test cases to Excel or CSV format.

In [ ]:
# Export function
def export_test_cases(test_cases_df, filename, format_type='excel'):
    """
    Export test cases to Excel or CSV format
    """
    try:
        if format_type.lower() == 'excel':
            filename_with_ext = f"{filename}.xlsx"
            
            # Create Excel writer with formatting
            with pd.ExcelWriter(filename_with_ext, engine='openpyxl') as writer:
                test_cases_df.to_excel(writer, sheet_name='Test Cases', index=False)
                
                # Get the workbook and worksheet
                workbook = writer.book
                worksheet = writer.sheets['Test Cases']
                
                # Auto-adjust column widths
                for column in worksheet.columns:
                    max_length = 0
                    column_letter = column[0].column_letter
                    
                    for cell in column:
                        try:
                            if len(str(cell.value)) > max_length:
                                max_length = len(str(cell.value))
                        except:
                            pass
                    
                    adjusted_width = min(max_length + 2, 50)  # Cap at 50 characters
                    worksheet.column_dimensions[column_letter].width = adjusted_width
                
            print(f"✅ Test cases exported to {filename_with_ext}")
            
        elif format_type.lower() == 'csv':
            filename_with_ext = f"{filename}.csv"
            test_cases_df.to_csv(filename_with_ext, index=False)
            print(f"✅ Test cases exported to {filename_with_ext}")
            
        else:
            print("❌ Unsupported format. Use 'excel' or 'csv'")
            return None
            
        return filename_with_ext
        
    except Exception as e:
        print(f"❌ Error exporting test cases: {e}")
        return None

# Export the test cases
exported_file = export_test_cases(df, OUTPUT_FILENAME, OUTPUT_FORMAT)

if exported_file:
    print(f"\n📄 File created: {exported_file}")
    print(f"📊 Total test cases: {len(df)}")
    print(f"🔢 Columns: {', '.join(df.columns.tolist())}")
    
    # Show summary statistics
    priority_counts = df['priority'].value_counts()
    print(f"\n📈 Priority distribution:")
    for priority, count in priority_counts.items():
        print(f"   {priority}: {count} test cases")

## Additional Test Case Generation

You can generate test cases for different requirements by modifying the requirement text below.

In [ ]:
# Example: Generate test cases for different requirements

additional_requirements = [
    "User can reset password by entering email address",
    "User profile information can be updated and saved",
    "File upload functionality supports PDF and image files"
]

print("🔄 Generating test cases for additional requirements...\n")

all_test_cases = []

for req in additional_requirements:
    print(f"Processing: {req}")
    cases = generator.generate_test_cases(req)
    all_test_cases.extend(cases)
    print(f"Generated {len(cases)} test cases\n")

# Combine with original test cases
combined_cases = test_cases + all_test_cases
combined_df = pd.DataFrame(combined_cases)

print(f"📊 Total test cases across all requirements: {len(combined_df)}")

# Export combined test cases
combined_filename = f"combined_test_cases_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
combined_file = export_test_cases(combined_df, combined_filename, OUTPUT_FORMAT)

if combined_file:
    print(f"\n✅ Combined test cases exported to: {combined_file}")

## Summary

This notebook demonstrates how to:

1. **Generate AI-powered test cases** using OpenAI or Hugging Face models
2. **Structure test cases** with proper columns (Test Case ID, Requirement, Step, Expected Result, Priority)
3. **Export results** to Excel or CSV format
4. **Handle multiple requirements** and combine results

### Key Benefits:
- ⏰ **Time-saving**: Automates manual test case writing
- 🎯 **Systematic**: Ensures comprehensive coverage
- 📋 **Structured**: Standardized output format
- 🔄 **Scalable**: Can handle multiple requirements
- 🛠️ **Flexible**: Supports both AI and rule-based generation

### Next Steps:
1. Customize the prompts for your specific domain
2. Add more sophisticated parsing for different requirement types
3. Integrate with test management tools
4. Add test case validation and quality metrics